قراءة البيانات

In [129]:
import pandas as pd
import numpy as np

df = pd.read_csv("dirty_cafe_sales.csv")

df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


فحس أنواع البيانات داخل الأعمدة

In [130]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


البحث عن القيم الغير صحيحة

In [131]:
df.isin(["ERROR", "UNKNOWN"]).sum()

Transaction ID        0
Item                636
Quantity            341
Price Per Unit      354
Total Spent         329
Payment Method      599
Location            696
Transaction Date    301
dtype: int64

البحث عن القيم الفارغة

In [132]:
df.isna().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

حل مشكلة أنواع البيانات داخل الأعمدة

In [133]:
df1 = df
df1["Quantity"] = pd.to_numeric(df1["Quantity"], errors="coerce")
df1["Price Per Unit"] = pd.to_numeric(df1["Price Per Unit"], errors="coerce")
df1["Total Spent"] = pd.to_numeric(df1["Total Spent"], errors="coerce")
df1["Transaction Date"] = pd.to_datetime(df1["Transaction Date"], errors="coerce")

In [134]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  object        
 1   Item              9667 non-null   object        
 2   Quantity          9521 non-null   float64       
 3   Price Per Unit    9467 non-null   float64       
 4   Total Spent       9498 non-null   float64       
 5   Payment Method    7421 non-null   object        
 6   Location          6735 non-null   object        
 7   Transaction Date  9540 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 625.1+ KB


التعامل مع القيم الخاطئة

In [135]:
df1.replace(["ERROR", "UNKNOWN", "nan", "<na>"], pd.NA, inplace=True)

In [136]:
df1.isin(["ERROR", "UNKNOWN", "nan", "<na>"]).sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

التعامل مع القيم الفارغة

تعبئة القيم الفارغة في عمود سعر الوحدة عن طريق حساب الفارق

In [137]:
mask = df1["Price Per Unit"].isna() & df1["Total Spent"].notna() & df1["Quantity"].notna() & (df1["Quantity"] != 0)
df1.loc[mask, "Price Per Unit"] = df1.loc[mask, "Total Spent"] / df1.loc[mask, "Quantity"]

In [138]:
df1.isna().sum()

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit        38
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

تعبئة بقية القيم الفارغة في عمود سعر الوحدة بسبب انه لم يتحقق الشرط مثل انه قد يكون الكمية فارغ او الإجمالي فارغ
وذلك عن طريق ال item

In [139]:
item_price_map = df.groupby("Item")["Price Per Unit"].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
)

mask = df["Price Per Unit"].isna() & df["Item"].notna()
df.loc[mask, "Price Per Unit"] = df.loc[mask, "Item"].map(item_price_map)

In [140]:
df1.isna().sum()

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit         6
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

حذف ما تبقى من عمود سعر الوحدة

In [141]:
df1 = df1.dropna(subset=["Price Per Unit"])

In [142]:
df1.isna().sum()

Transaction ID         0
Item                 963
Quantity             476
Price Per Unit         0
Total Spent          499
Payment Method      3175
Location            3958
Transaction Date     460
dtype: int64

تعبئة القيم الفارغة في عمود الكمية عن طريق قسمة السعر الإجمالي على سعر الوحدة

In [143]:
mask = df1["Quantity"].isna() & df1["Total Spent"].notna() & df1["Price Per Unit"].notna()

df1.loc[mask, "Quantity"] = df1.loc[mask, "Total Spent"] / df1.loc[mask, "Price Per Unit"]

In [144]:
df1.isna().sum()

Transaction ID         0
Item                 963
Quantity              20
Price Per Unit         0
Total Spent          499
Payment Method      3175
Location            3958
Transaction Date     460
dtype: int64

تعبئة بقية القيم الفارغة بالمنوال عن طريق ال item يعني كم غالبا بيأخوا من هذا المنتج

In [145]:
item_quantity_map = df1.groupby("Item")["Quantity"].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
)

mask = df1["Quantity"].isna() & df1["Item"].notna()
df1.loc[mask, "Quantity"] = df1.loc[mask, "Item"].map(item_quantity_map)

In [146]:
df1.isna().sum()

Transaction ID         0
Item                 963
Quantity               0
Price Per Unit         0
Total Spent          499
Payment Method      3175
Location            3958
Transaction Date     460
dtype: int64

تعبئة القيم الفارغة في عمود الإجمالي عن طريق حساب الضرب

In [147]:
mask = df1["Total Spent"].isna() & df1["Quantity"].notna() & df1["Price Per Unit"].notna()
df1.loc[mask, "Total Spent"] = df1.loc[mask, "Quantity"] * df1.loc[mask, "Price Per Unit"]

In [148]:
df1.isna().sum()

Transaction ID         0
Item                 963
Quantity               0
Price Per Unit         0
Total Spent            0
Payment Method      3175
Location            3958
Transaction Date     460
dtype: int64

تعبئة القيم الفارغة في عمود ال item عن طريق سعر الوحدة

In [149]:
price_item_map = df1.groupby("Price Per Unit")["Item"].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA
)

mask = df1["Item"].isna() & df1["Price Per Unit"].notna()
df1.loc[mask, "Item"] = df1.loc[mask, "Price Per Unit"].map(price_item_map)

In [150]:
df1.isna().sum()

Transaction ID         0
Item                   0
Quantity               0
Price Per Unit         0
Total Spent            0
Payment Method      3175
Location            3958
Transaction Date     460
dtype: int64

تعبئة القيم الفارغة في عمود المنطقة وعمود طريقة الدفع

In [151]:
df1["Payment Method"] = df1["Payment Method"].fillna("Unknown")
df1["Location"] = df1["Location"].fillna("Unknown")

C:\Users\abdos\AppData\Local\Temp\ipykernel_3544\550902157.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1["Payment Method"] = df1["Payment Method"].fillna("Unknown")
C:\Users\abdos\AppData\Local\Temp\ipykernel_3544\550902157.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1["Location"] = df1["Location"].fillna("Unknown")


In [152]:
df1.isna().sum()

Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    460
dtype: int64

تعبئة الخلايا الفارغة في التاريخ بحسب الخلية الي قبلها

In [153]:
df1["Transaction Date"] = df1["Transaction Date"].ffill()

C:\Users\abdos\AppData\Local\Temp\ipykernel_3544\2948808887.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1["Transaction Date"] = df1["Transaction Date"].ffill()


In [154]:
df1.isna().sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

تنظيف النصوص (lower + strip)

In [155]:
text_cols = ["Item", "Payment Method", "Location"]

for col in text_cols:
    df1[col] = df1[col].astype(str).str.lower().str.strip()

C:\Users\abdos\AppData\Local\Temp\ipykernel_3544\3011606440.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1[col] = df1[col].astype(str).str.lower().str.strip()
C:\Users\abdos\AppData\Local\Temp\ipykernel_3544\3011606440.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1[col] = df1[col].astype(str).str.lower().str.strip()
C:\Users\abdos\AppData\Local\Temp\ipykernel_3544\3011606440.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row

التأكد من القيم داخل كل عمود نصي

In [160]:
df1['Item'].value_counts()

Item
juice       1418
sandwich    1358
coffee      1291
salad       1272
cookie      1213
tea         1207
cake        1139
smoothie    1096
Name: count, dtype: int64

In [161]:
df1['Payment Method'].value_counts()

Payment Method
unknown           3175
digital wallet    2290
credit card       2271
cash              2258
Name: count, dtype: int64

In [162]:
df1['Location'].value_counts()

Location
unknown     3958
takeaway    3022
in-store    3014
Name: count, dtype: int64

إنشاء عمود season من التاريخ

In [156]:
def get_season(date):
    month = date.month
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"

df1["season"] = df1["Transaction Date"].apply(get_season)

C:\Users\abdos\AppData\Local\Temp\ipykernel_3544\1394131146.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1["season"] = df1["Transaction Date"].apply(get_season)


In [157]:
df1.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,season
0,TXN_1961373,coffee,2.0,2.0,4.0,credit card,takeaway,2023-09-08,Fall
1,TXN_4977031,cake,4.0,3.0,12.0,cash,in-store,2023-05-16,Spring
2,TXN_4271903,cookie,4.0,1.0,4.0,credit card,in-store,2023-07-19,Summer
3,TXN_7034554,salad,2.0,5.0,10.0,unknown,unknown,2023-04-27,Spring
4,TXN_3160411,coffee,2.0,2.0,4.0,digital wallet,in-store,2023-06-11,Summer


حفظ البيانات

In [158]:
df.to_csv("cleaned_cafe_sales.csv", index=False)
"saved cleaned_cafe_sales.csv"

'saved cleaned_cafe_sales.csv'